# MSNet Scout Runs: Washington RGB-D Object Fine-tuning Convergence Analysis

**Purpose:** Run 3-5 training runs on the Washington RGB-D Object dataset (51 categories) with different learning rates to observe convergence behavior. All runs share the same config except LR, and use ReduceOnPlateau to adapt LR automatically.

Helps decide the optimal epoch budget for full HPO and final training.

---

## Checklist Before Running:

- [ ] **Enable A100 GPU:** Runtime > Change runtime type > A100
- [ ] **Washington dataset on Drive:** `Shareddrives/MSNN-Capstone/data/washington_rgbd_256.tar.gz` (produced by `notebooks/washington_preprocess.ipynb`)

## 1. Environment Setup & GPU Verification

In [ ]:
# Check GPU availability and specs
import torch
import subprocess

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\nA100 GPU detected - optimal for training")
    elif 'V100' in gpu_name:
        print("\nV100 GPU detected - good for training (slower than A100)")
    elif 'T4' in gpu_name:
        print("\nT4 GPU detected - will be slower, consider upgrading to A100")
    else:
        print(f"\nGPU: {gpu_name}")
else:
    print("\nNO GPU DETECTED!")
    print("Enable GPU: Runtime -> Change runtime type -> Hardware accelerator: GPU")
    raise RuntimeError("GPU is required for training")

print("\n" + "=" * 60)

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

print("\nGoogle Drive mounted successfully!")
print(f"\nDrive contents:")
!ls -la /content/drive/Shareddrives/MSNN-Capstone/ | head -20

## 3. Clone Repository to Local Disk (Fast I/O)

In [ ]:
import os
from pathlib import Path

# Configuration
PROJECT_NAME = "Multi-Stream-Neural-Networks"
GITHUB_REPO = "https://github.com/clingergab/Multi-Stream-Neural-Networks.git"
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"

print("=" * 60)
print("REPOSITORY SETUP")
print("=" * 60)

os.chdir('/content')

if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"Repo already exists: {LOCAL_REPO_PATH}")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
else:
    if Path(LOCAL_REPO_PATH).exists():
        !rm -rf {LOCAL_REPO_PATH}
    print(f"Cloning from {GITHUB_REPO}...")
    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}
    if not Path(LOCAL_REPO_PATH).exists():
        raise RuntimeError(f"Failed to clone repository")
    os.chdir(LOCAL_REPO_PATH)

print(f"\nWorking directory: {os.getcwd()}")
!ls -la {LOCAL_REPO_PATH}
print("\n" + "=" * 60)

## 4. Install Dependencies

In [ ]:
# Install required packages
print("Installing dependencies...")

!pip install -q h5py tqdm matplotlib seaborn ray[tune] kornia thop

# Verify installations
import h5py
import tqdm
import matplotlib
import seaborn
import kornia
import thop

print("All dependencies installed!")
print(f"   h5py: {h5py.__version__}")
print(f"   matplotlib: {matplotlib.__version__}")
print(f"   kornia: {kornia.__version__}")
print(f"   thop: {thop.__version__}")

## 5. Setup Python Path & Imports

In [ ]:
import sys
import os
import json
import warnings
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

# Remove cached modules
modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_reload:
    del sys.modules[module]

# Add project to Python path
project_root = '/content/Multi-Stream-Neural-Networks'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Model
from src.models.linear_integration.ms_net import ms_resnet18

# Dataset
from src.data_utils.washington_dataset import (
    WashingtonRGBDDataset,
    get_washington_dataloaders,
    _load_norm_stats,
)

# Training
from src.training.augmentation_config import AugmentationConfig
from src.training.optimizers import create_stream_optimizer
from src.training.schedulers import setup_scheduler
from src.utils.seed import set_seed

print("All imports successful!")

In [ ]:
SEED = 152
DETERMINISTIC = False
set_seed(SEED, deterministic=DETERMINISTIC)
print(f"Seed: {SEED}, Deterministic: {DETERMINISTIC}")

## 6. Copy Washington RGB-D Dataset to RAM

In [ ]:
DRIVE_WASHINGTON_TAR = "/content/drive/Shareddrives/MSNN-Capstone/data/washington_rgbd_256.tar.gz"
WASHINGTON_DATA_PATH = "/dev/shm/washington_rgbd_256"

print("=" * 60)
print("WASHINGTON RGB-D OBJECT DATASET SETUP (51 CATEGORIES)")
print("=" * 60)

if Path(WASHINGTON_DATA_PATH).exists():
    print(f"Already on local disk: {WASHINGTON_DATA_PATH}")
elif Path(DRIVE_WASHINGTON_TAR).exists():
    print(f"Found on Drive: {DRIVE_WASHINGTON_TAR}")
    tar_name = Path(DRIVE_WASHINGTON_TAR).name
    local_tar = f"/dev/shm/{tar_name}"
    !rsync -ah --info=progress2 "{DRIVE_WASHINGTON_TAR}" {local_tar}
    print(f"\nExtracting...")
    !tar -xzf {local_tar} -C /dev/shm/ 2>&1 | grep -v "Ignoring unknown extended header"
    !rm {local_tar}
    print("Extracted.")
else:
    raise FileNotFoundError(f"Dataset not found at {DRIVE_WASHINGTON_TAR}")

# Verify: count paired *_rgb.pt samples per split (each has a sibling *_depth.pt)
dataset_root = Path(WASHINGTON_DATA_PATH)
for split in ['train', 'val', 'test']:
    split_dir = dataset_root / split
    if split_dir.exists():
        n = len(list(split_dir.glob('*/*_rgb.pt')))
        print(f"  {split}: {n} samples")

# Metadata files written by the preprocess notebook
for meta in ['class_names.txt', 'norm_stats.json']:
    meta_path = dataset_root / meta
    status = 'OK' if meta_path.exists() else 'MISSING'
    print(f"  {meta}: {status}")

print(f"\nWashington dataset ready at: {WASHINGTON_DATA_PATH}")
print("=" * 60)

## 7. Scout Run Configuration

All runs share the same config **except learning rates**.
ReduceOnPlateau scheduler adapts LR automatically based on val MCA plateau.

In [ ]:
STREAM_LABELS = {0: 'RGB', 1: 'Depth'}

# ======================== DATASET ========================
DATASET_CONFIG = {
    'data_root': WASHINGTON_DATA_PATH,
    'batch_size': 64,
    'num_workers': 5,
    'crop_size': 224,
    'seed': SEED,
}

AUGMENTATION_CONFIG = AugmentationConfig(
    rgb_aug_prob=1.0,      # TODO: fill from HPO
    rgb_aug_mag=1.0,       # TODO: fill from HPO
    depth_aug_prob=1.0,    # TODO: fill from HPO
    depth_aug_mag=1.0,     # TODO: fill from HPO
)

# ======================== MODEL (shared across all runs) ========================
# 51 Washington RGB-D Object categories.
MODEL_CONFIG = {
    'num_classes': 51,
    'stream_input_channels': [3, 1],  # RGB + Depth
    'width_multiplier': 1.0,          # Full-width MSNet (training from random init)
    'dropout_p': 0.5,                 # TODO: fill
    'device': 'cuda',
    'use_amp': True,
}

# ======================== SHARED TRAINING CONFIG ========================
TRAIN_CONFIG = {
    'epochs': 150,
    'grad_clip_norm': 1.0,            # TODO: fill
    'label_smoothing': 0.1,           # TODO: fill
    'modality_dropout': True,
    'modality_dropout_start': 5,
    'modality_dropout_ramp': 20,
    'modality_dropout_rate': 0.1,     # TODO: fill
    'stream_monitoring': True,
    'gradient_monitoring': False,      # Off for speed in scout runs
    'track_integration_weights': False,
}

# ======================== PLATEAU SCHEDULER CONFIG ========================
SCHEDULER_CONFIG = {
    'scheduler_type': 'plateau',
    'mode': 'max',                    # Monitor val_mca (higher is better)
    'scheduler_patience': 8,          # Epochs to wait before reducing LR
    'factor': 0.5,                    # Halve LR on plateau
    'min_lr': 1e-7,
    'warmup_epochs': 5,
    'warmup_start_factor': 0.2,
}

# ======================== OPTIMIZER CONFIG (shared, except LRs) ========================
OPTIMIZER_BASE_CONFIG = {
    'optimizer_type': 'adamw',
    'stream_weight_decays': [5e-5, 5e-5],  # TODO: fill
    'integration_weight_decay': 1e-4,       # TODO: fill
    'stem_lr_multiplier': 1.0,  # >1.0 boosts stem (conv1) LR; 1.0 = no change
}

# ======================== SCOUT RUNS ========================
# Each run gets a different LR. Format: (label, stream_lrs, shared_lr)
# Tip: span ~1 order of magnitude to bracket the optimal range.
SCOUT_RUNS = [
    ("LR=5e-5",  [5e-5, 5e-5],   5e-5),
    ("LR=1e-4",  [1e-4, 1e-4],   1e-4),
    ("LR=3e-4",  [3e-4, 3e-4],   3e-4),
    ("LR=7e-4",  [7e-4, 7e-4],   7e-4),
    ("LR=1.5e-3", [1.5e-3, 1.5e-3], 1.5e-3),
]

# ======================== OUTPUT PATHS ========================
OUTPUT_DIR = Path("/content/drive/Shareddrives/MSNN-Capstone/msnet_checkpoints/washington_scout")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Scout runs: {len(SCOUT_RUNS)}")
for label, slrs, slr in SCOUT_RUNS:
    print(f"  {label}: stream_lrs={slrs}, shared_lr={slr}")
print(f"Epochs per run: {TRAIN_CONFIG['epochs']}")
print(f"Num classes: {MODEL_CONFIG['num_classes']}")
print(f"Scheduler: {SCHEDULER_CONFIG['scheduler_type']} (patience={SCHEDULER_CONFIG['scheduler_patience']}, factor={SCHEDULER_CONFIG['factor']})")
print(f"Output dir: {OUTPUT_DIR}")

## 8. Load Dataset

In [ ]:
print("=" * 60)
print("LOADING WASHINGTON RGB-D OBJECT DATASET")
print("=" * 60)

# Load normalization stats for GPU-side normalization inside model.compile.
norm_stats = _load_norm_stats(DATASET_CONFIG['data_root'])

# The preprocess pipeline already produced train/val/test splits on disk,
# so we just ask the factory for all three loaders. It wires in:
#   - WeightedRandomSampler (balanced_sampling=True)
#   - persistent_workers / pin_memory
#   - a deterministic worker_init_fn seeded from `seed`
#   - crop_size=224 RandomCrop on train, CenterCrop on val/test
# normalize=False here because we normalize on GPU after augmentation.
train_loader, val_loader, test_loader, num_classes = get_washington_dataloaders(
    data_root=DATASET_CONFIG['data_root'],
    batch_size=DATASET_CONFIG['batch_size'],
    num_workers=DATASET_CONFIG['num_workers'],
    crop_size=DATASET_CONFIG['crop_size'],
    seed=DATASET_CONFIG['seed'],
    normalize=False,
    balanced_sampling=True,
    **AUGMENTATION_CONFIG.to_dict(),
)

assert num_classes == MODEL_CONFIG['num_classes'], (
    f"num_classes mismatch: dataset has {num_classes}, "
    f"MODEL_CONFIG expects {MODEL_CONFIG['num_classes']}. "
    f"Update MODEL_CONFIG['num_classes'] or re-run preprocessing."
)

# Quick class-distribution sanity check on the train split.
train_dist = train_loader.dataset.get_class_distribution()
train_counts = [v['count'] for v in train_dist.values()]
print(
    f"  Classes: {num_classes} | "
    f"train per-class count min={min(train_counts)} "
    f"median={int(np.median(train_counts))} max={max(train_counts)}"
)

# Optional: peek at a batch to confirm shapes/dtypes line up with the model.
# rgb_batch, depth_batch, label_batch = next(iter(train_loader))
# print(f"  Batch shapes: RGB={rgb_batch.shape}, Depth={depth_batch.shape}, Labels={label_batch.shape}")
print("=" * 60)

## 9. Run All Scout Experiments

Each run: fresh MSNet from random init -> train up to 150 epochs with ReduceOnPlateau on Washington val MCA.

In [ ]:
warnings.filterwarnings(
    'ignore',
    message='The epoch parameter in `scheduler.step\\(\\)` was not necessary',
    category=UserWarning
)

all_histories = {}

for run_idx, (label, stream_lrs, shared_lr) in enumerate(SCOUT_RUNS):
    print("\n" + "=" * 70)
    print(f"  SCOUT RUN {run_idx + 1}/{len(SCOUT_RUNS)}: {label}")
    print("=" * 70)

    # Reset seed for reproducibility across runs
    set_seed(SEED, deterministic=DETERMINISTIC)

    # Fresh model (random init)
    model = ms_resnet18(
        num_classes=MODEL_CONFIG['num_classes'],
        stream_input_channels=MODEL_CONFIG['stream_input_channels'],
        width_multiplier=MODEL_CONFIG['width_multiplier'],
        dropout_p=MODEL_CONFIG['dropout_p'],
        device=MODEL_CONFIG['device'],
        use_amp=MODEL_CONFIG['use_amp'],
    )

    # Create optimizer with this run's LRs
    optimizer = create_stream_optimizer(
        model,
        optimizer_type=OPTIMIZER_BASE_CONFIG['optimizer_type'],
        stream_lrs=stream_lrs,
        stream_weight_decays=OPTIMIZER_BASE_CONFIG['stream_weight_decays'],
        shared_lr=shared_lr,
        integration_weight_decay=OPTIMIZER_BASE_CONFIG['integration_weight_decay'],
        stem_lr_multiplier=OPTIMIZER_BASE_CONFIG['stem_lr_multiplier'],
    )

    # Create ReduceOnPlateau scheduler
    scheduler = setup_scheduler(
        optimizer,
        scheduler_type=SCHEDULER_CONFIG['scheduler_type'],
        mode=SCHEDULER_CONFIG['mode'],
        scheduler_patience=SCHEDULER_CONFIG['scheduler_patience'],
        factor=SCHEDULER_CONFIG['factor'],
        min_lr=SCHEDULER_CONFIG['min_lr'],
        warmup_epochs=SCHEDULER_CONFIG['warmup_epochs'],
        warmup_start_factor=SCHEDULER_CONFIG['warmup_start_factor'],
    )

    # Compile
    model.compile(
        optimizer=optimizer,
        scheduler=scheduler,
        loss='cross_entropy',
        label_smoothing=TRAIN_CONFIG['label_smoothing'],
        gpu_augmentation=True,
        norm_stats=norm_stats,
        **AUGMENTATION_CONFIG.to_dict(),
    )

    # Train with val split for plateau scheduler + monitoring
    history = model.fit(
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=TRAIN_CONFIG['epochs'],
        verbose=True,
        early_stopping=True,
        patience=15,
        grad_clip_norm=TRAIN_CONFIG['grad_clip_norm'],
        stream_monitoring=TRAIN_CONFIG['stream_monitoring'],
        monitor='val_mca',
        modality_dropout=TRAIN_CONFIG['modality_dropout'],
        modality_dropout_start=TRAIN_CONFIG['modality_dropout_start'],
        modality_dropout_ramp=TRAIN_CONFIG['modality_dropout_ramp'],
        modality_dropout_rate=TRAIN_CONFIG['modality_dropout_rate'],
        gradient_monitoring=TRAIN_CONFIG['gradient_monitoring'],
        track_integration_weights=TRAIN_CONFIG['track_integration_weights'],
    )

    all_histories[label] = history

    # Print final metrics
    final_train_acc = history['train_accuracy'][-1] * 100
    final_val_acc = history['val_accuracy'][-1] * 100
    final_val_mca = history.get('val_mca', [0])[-1] * 100
    final_lr = history['learning_rates'][-1]
    print(f"\n  Final: Train Acc={final_train_acc:.1f}%  Val Acc={final_val_acc:.1f}%  Val MCA={final_val_mca:.1f}%  LR={final_lr:.2e}")

    # Free GPU memory for next run
    del model, optimizer, scheduler
    torch.cuda.empty_cache()

print("\n" + "=" * 70)
print("  ALL SCOUT RUNS COMPLETE")
print("=" * 70)

## 10. Convergence Comparison

Overlay all scout runs to compare convergence speed, final accuracy, and plateau behavior.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 11))

colors = plt.cm.tab10(np.linspace(0, 1, len(all_histories)))

# --- Plot 1: Train Loss ---
ax = axes[0, 0]
for (label, history), color in zip(all_histories.items(), colors):
    ax.plot(history['train_loss'], label=label, color=color, linewidth=1.5)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Train Loss', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Plot 2: Val Loss ---
ax = axes[0, 1]
for (label, history), color in zip(all_histories.items(), colors):
    if 'val_loss' in history and history['val_loss']:
        ax.plot(history['val_loss'], label=label, color=color, linewidth=1.5)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Val Loss', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Plot 3: Train Accuracy ---
ax = axes[0, 2]
for (label, history), color in zip(all_histories.items(), colors):
    ax.plot([a*100 for a in history['train_accuracy']], label=label, color=color, linewidth=1.5)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Train Accuracy', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Plot 4: Val Accuracy ---
ax = axes[1, 0]
for (label, history), color in zip(all_histories.items(), colors):
    if 'val_accuracy' in history and history['val_accuracy']:
        ax.plot([a*100 for a in history['val_accuracy']], label=label, color=color, linewidth=1.5)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Val Accuracy', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Plot 5: Val MCA ---
ax = axes[1, 1]
for (label, history), color in zip(all_histories.items(), colors):
    if 'val_mca' in history and history['val_mca']:
        ax.plot([m*100 for m in history['val_mca']], label=label, color=color, linewidth=1.5)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MCA (%)', fontsize=12)
ax.set_title('Val Mean Class Accuracy', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Plot 6: Learning Rate ---
ax = axes[1, 2]
for (label, history), color in zip(all_histories.items(), colors):
    ax.plot(history['learning_rates'], label=label, color=color, linewidth=1.5)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Learning Rate', fontsize=12)
ax.set_yscale('log')
ax.set_title('Learning Rate (Plateau Reductions)', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
convergence_path = OUTPUT_DIR / "scout_convergence.pdf"
plt.savefig(convergence_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"Convergence plots saved to: {convergence_path}")

## 11. Results Summary

In [ ]:
print("=" * 90)
print(f"{'Run':<14} {'Best Val MCA':>14} {'Best Val Acc':>14} {'Final Train Acc':>16} {'Plateau Hits':>13} {'Best Epoch':>11}")
print("-" * 90)

for label, history in all_histories.items():
    val_mcas = history.get('val_mca', [])
    val_accs = history.get('val_accuracy', [])
    train_accs = history['train_accuracy']
    lrs = history['learning_rates']

    best_mca = max(val_mcas) * 100 if val_mcas else 0
    best_mca_epoch = val_mcas.index(max(val_mcas)) + 1 if val_mcas else 0
    best_acc = max(val_accs) * 100 if val_accs else 0
    final_train = train_accs[-1] * 100

    # Count plateau reductions (LR drops)
    plateau_hits = sum(1 for i in range(1, len(lrs)) if lrs[i] < lrs[i-1] * 0.9)

    print(f"{label:<14} {best_mca:>13.2f}% {best_acc:>13.2f}% {final_train:>15.2f}% {plateau_hits:>13} {best_mca_epoch:>11}")

print("=" * 90)

# Convergence analysis
print("\nConvergence Analysis:")
for label, history in all_histories.items():
    val_mcas = history.get('val_mca', [])
    if not val_mcas:
        continue
    best_mca = max(val_mcas)
    threshold_95 = best_mca * 0.95
    for epoch, mca in enumerate(val_mcas):
        if mca >= threshold_95:
            print(f"  {label}: Reaches 95% of peak MCA at epoch {epoch + 1}")
            break

## 12. Save Scout Results

In [ ]:
# Save all histories for later analysis
scout_results = {}
for label, history in all_histories.items():
    scout_results[label] = {
        'train_loss': [float(x) for x in history['train_loss']],
        'train_accuracy': [float(x) for x in history['train_accuracy']],
        'train_mca': [float(x) for x in history.get('train_mca', [])],
        'val_loss': [float(x) for x in history.get('val_loss', [])],
        'val_accuracy': [float(x) for x in history.get('val_accuracy', [])],
        'val_mca': [float(x) for x in history.get('val_mca', [])],
        'learning_rates': [float(x) for x in history['learning_rates']],
        # Per-stream and stem LR curves (when stream_monitoring=True)
        **{f'stream_{i}_lr': [float(x) for x in history.get(f'stream_{i}_lr', [])]
           for i in range(2)},
        **{f'stem_{i}_lr': [float(x) for x in history.get(f'stem_{i}_lr', [])]
           for i in range(2) if f'stem_{i}_lr' in history},
    }

scout_path = OUTPUT_DIR / "scout_results.json"
with open(scout_path, 'w') as f:
    json.dump({
        'config': {
            'dataset_config': DATASET_CONFIG,
            'model_config': MODEL_CONFIG,
            'train_config': TRAIN_CONFIG,
            'scheduler_config': SCHEDULER_CONFIG,
            'optimizer_base_config': OPTIMIZER_BASE_CONFIG,
            'augmentation_config': AUGMENTATION_CONFIG.to_dict(),
            'seed': SEED,
        },
        'runs': scout_results,
    }, f, indent=2)

print(f"Scout results saved: {scout_path}")